<a href="https://colab.research.google.com/github/AbeeraImran/GEN-AI_Seq2Seq_UrduQA/blob/main/code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
!pip install datasets sentencepiece sacrebleu rouge-score torch

In [23]:
from datasets import load_dataset

ds = load_dataset("uqa/UQA")

ex = ds["train"][0]
print("Keys:", ex.keys())
print("Question:", ex["question"])
print("Answer Data:", ex.get("answer") or ex.get("answers"))

n_total = len(ds["train"])
n_ans = sum(not a["is_impossible"] for a in ds["train"])
print(f"train rows: {n_total}, answerable: {n_ans}")

Keys: dict_keys(['id', 'title', 'context', 'question', 'is_impossible', 'answer', 'answer_start'])
Question: بیونس نے کب مقبولیت حاصل کرنا شروع کی؟
Answer Data: 1990 کی دہائی کے آخر میں
train rows: 124745, answerable: 83018


In [24]:
import csv

ANS_OPEN, ANS_CLOSE = "", ""
SENT_DELIMS = "\u06D4\u061F!"

def split_sentences(text):
    start = 0
    for i, ch in enumerate(text):
        if ch in SENT_DELIMS:
            yield start, i + 1, text[start:i + 1]
            start = i + 1
    if start < len(text):
        yield start, len(text), text[start:]

def make_pair(example, max_src=60, max_tgt=25):
    if example.get("is_impossible", False):
        return None

    ans_data = example.get("answer") or example.get("answers")
    start_data = example.get("answer_start")

    try:
        if isinstance(ans_data, dict):
            a_text = ans_data["text"][0]
            a_start = ans_data["answer_start"][0]
        else:
            a_text = ans_data[0] if isinstance(ans_data, list) else ans_data
            a_start = start_data[0] if isinstance(start_data, list) else start_data
    except (IndexError, KeyError, TypeError):
        return None

    context = example["context"]

    for s, e, sent in split_sentences(context):
        if s <= a_start < e:
            rel = a_start - s
            if sent[rel:rel + len(a_text)] != a_text:
                return None

            src = (sent[:rel] + " " + ANS_OPEN + " " + a_text + " "
                   + ANS_CLOSE + " " + sent[rel + len(a_text):]).strip()
            src = " ".join(src.split())
            tgt = " ".join(example["question"].split())

            if len(src.split()) > max_src or len(tgt.split()) > max_tgt:
                return None
            return src, tgt
    return None

def build_split(split, out_path):
    pairs = [p for p in map(make_pair, split) if p is not None]
    with open(out_path, "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f, delimiter="\t", quoting=csv.QUOTE_NONE, escapechar="\\")
        w.writerows(pairs)
    print(f"{out_path}: {len(pairs)} pairs")
    return pairs

train_pairs = build_split(ds["train"], "train.tsv")
valid_pairs = build_split(ds["validation"], "valid.tsv")

train.tsv: 75871 pairs
valid.tsv: 10129 pairs


In [25]:
import sentencepiece as spm

ANS_OPEN, ANS_CLOSE = "<ans>", "</ans>"

with open("sp_corpus.txt", "w", encoding="utf-8") as f:
    for src, tgt in train_pairs:
        f.write(src + "\n" + tgt + "\n")

spm.SentencePieceTrainer.train(
    input="sp_corpus.txt",
    model_prefix="ur_sp",
    vocab_size=8000,
    model_type="unigram",
    character_coverage=1.0,
    user_defined_symbols=[ANS_OPEN, ANS_CLOSE],
    pad_id=0, unk_id=1, bos_id=2, eos_id=3,
)

sp = spm.SentencePieceProcessor(model_file="ur_sp.model")
PAD, UNK, BOS, EOS = 0, 1, 2, 3

src, tgt = train_pairs[0]
print("Tokens:", sp.encode(src, out_type=str))
print("IDs:", sp.encode(tgt))
print("Match Check:", sp.decode(sp.encode(tgt)) == tgt)

Tokens: ['▁ہیوسٹن', '▁،', '▁ٹیکساس', '▁میں', '▁پیدا', '▁ہوئی', '▁اور', '▁اس', '▁کی', '▁پرورش', '▁ہوئی', '▁،', '▁اس', '▁نے', '▁بچپن', '▁میں', '▁مختلف', '▁گانے', '▁اور', '▁رقص', '▁کے', '▁مقابلوں', '▁میں', '▁پرفارم', '▁کیا', '▁،', '▁اور', '▁1990', '▁کی', '▁دہائی', '▁کے', '▁آخر', '▁میں', '▁R', '&', 'B', '▁گر', 'ل', '▁گروپ', '▁ڈسٹنی', '▁چائلڈ', '▁کے', '▁لیڈ', '▁گلوکار', '▁کی', '▁حیثیت', '▁سے', '▁شہر', 'ت', '▁حاصل', '▁کی۔']
IDs: [2776, 17, 83, 2834, 103, 150, 99, 8, 10]
Match Check: True


Dataset class

In [26]:
import torch
from torch.utils.data import Dataset, DataLoader

class QGDataset(Dataset):
    def __init__(self, tsv_path, sp_model):
        self.sp = sp_model
        self.pairs = []

        with open(tsv_path, 'r', encoding='utf-8') as file:
            for line in file:
                line = line.strip()
                if not line:
                    continue

                parts = line.split('	')
                if len(parts) == 2:
                    self.pairs.append((parts[0], parts[1]))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src_text, tgt_text = self.pairs[idx]

        src_ids = self.sp.encode(src_text)

        tgt_ids = [2] + self.sp.encode(tgt_text) + [3]

        return torch.tensor(src_ids, dtype=torch.long), torch.tensor(tgt_ids, dtype=torch.long)

In [27]:
train_dataset = QGDataset("train.tsv", sp)
print("Total dataset size:", len(train_dataset))
src_sample, tgt_sample = train_dataset[0]
print("Source tensor:", src_sample)
print("Target tensor:", tgt_sample)

Total dataset size: 75871
Source tensor: tensor([1164,    9, 2656,    7,  187,  144,   11,   20,    8, 5921,  144,    9,
          20,   17, 3960,    7,  165, 1242,   11, 3244,    6, 4458,    7, 2635,
          15,    9,   11, 1114,    8,  167,    6,  318,    7, 1455, 6854, 1802,
         672,  102,  196, 5256, 4158,    6, 2420, 3056,    8,  404,   12,   75,
         151,  103,  175])
Target tensor: tensor([   2, 2776,   17,   83, 2834,  103,  150,   99,    8,   10,    3])


In [34]:
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Subset

def collate_fn(batch):
    src_list = [item[0] for item in batch]
    tgt_list = [item[1] for item in batch]
    src_lengths = torch.tensor([len(s) for s in src_list], dtype=torch.long)
    padded_src = pad_sequence(src_list, batch_first=True, padding_value=0)
    padded_tgt = pad_sequence(tgt_list, batch_first=True, padding_value=0)
    return padded_src, padded_tgt, src_lengths

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    collate_fn=collate_fn
)

valid_dataset = QGDataset("valid.tsv", sp)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=64,
    shuffle=False,
    collate_fn=collate_fn
)

Encoder:

In [35]:
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class Encoder(nn.Module):
    def __init__(self, vocab_size=8000, emb_dim=256, hidden_size=512, num_layers=2, dropout=0.3):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)

        self.rnn = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            bidirectional=True,
            dropout=dropout,
            batch_first=True
        )

    def forward(self, src, src_lengths):
        embedded = self.embedding(src)

        packed = pack_padded_sequence(
            embedded,
            src_lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        outputs, (hidden, cell) = self.rnn(packed)

        outputs, _ = pad_packed_sequence(outputs, batch_first=True)

        return outputs, hidden, cell

In [36]:
encoder = Encoder()

batch_src, batch_tgt, batch_lens = next(iter(train_loader))

enc_outputs, enc_hidden, enc_cell = encoder(batch_src, batch_lens)

print("Encoder Output Shape:", enc_outputs.shape)
print("Hidden State Shape:", enc_hidden.shape)
print("Cell State Shape:", enc_cell.shape)

Encoder Output Shape: torch.Size([64, 98, 1024])
Hidden State Shape: torch.Size([4, 64, 512])
Cell State Shape: torch.Size([4, 64, 512])


Decoder:

In [37]:
import torch.nn as nn

class DecoderBase(nn.Module):
    def __init__(self, vocab_size=8000, emb_dim=256, hidden_size=512, num_layers=2, dropout=0.3):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)

        self.rnn = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            batch_first=True
        )

        self.out = nn.Linear(hidden_size, vocab_size)

    def forward(self, input_token, hidden, cell):
        embedded = self.embedding(input_token)

        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))

        prediction = self.out(output.squeeze(1))

        return prediction, hidden, cell

training loop

In [40]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = Encoder().to(device)
decoder = DecoderBase().to(device)

params = list(encoder.parameters()) + list(decoder.parameters())
optimizer = optim.Adam(params, lr=0.001)

criterion = nn.CrossEntropyLoss(ignore_index=0)

TEACHER_FORCING_RATIO = 0.5

def train_one_epoch(encoder, decoder, loader, optimizer, criterion):
    encoder.train()
    decoder.train()
    total_loss = 0

    for src, tgt, src_lengths in loader:
        src, tgt, src_lengths = src.to(device), tgt.to(device), src_lengths.to(device)
        optimizer.zero_grad()

        encoder_outputs, hidden, cell = encoder(src, src_lengths)

        batch_size, tgt_len = tgt.shape

        hidden = hidden.view(2, 2, batch_size, 512)
        hidden = hidden[:, 0, :, :] + hidden[:, 1, :, :]

        cell = cell.view(2, 2, batch_size, 512)
        cell = cell[:, 0, :, :] + cell[:, 1, :, :]

        loss = 0
        input_token = tgt[:, 0].unsqueeze(1)

        for t in range(1, tgt_len):
            prediction, hidden, cell = decoder(input_token, hidden, cell)
            loss += criterion(prediction, tgt[:, t])

            if random.random() < TEACHER_FORCING_RATIO:
                input_token = tgt[:, t].unsqueeze(1)
            else:
                input_token = prediction.argmax(1).unsqueeze(1)

        loss = loss / (tgt_len - 1)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [42]:
import torch

EPOCHS = 15
best_valid_loss = float('inf')

def evaluate_loss(encoder, decoder, valid_loader, criterion):
    encoder.eval()
    decoder.eval()
    epoch_loss = 0

    with torch.no_grad():
        for src, tgt, src_lengths in valid_loader:
            src, tgt = src.to(device), tgt.to(device)

            encoder_outputs, hidden, cell = encoder(src, src_lengths)

            batch_size, tgt_len = tgt.shape
            hidden = hidden.view(2, 2, batch_size, 512)
            hidden = hidden[:, 0, :, :] + hidden[:, 1, :, :]
            cell = cell.view(2, 2, batch_size, 512)
            cell = cell[:, 0, :, :] + cell[:, 1, :, :]

            input_token = tgt[:, 0].unsqueeze(1)
            loss = 0

            for t in range(1, tgt_len):
                prediction, hidden, cell = decoder(input_token, hidden, cell)
                loss += criterion(prediction, tgt[:, t])
                input_token = tgt[:, t].unsqueeze(1)

            loss = loss / (tgt_len - 1)
            epoch_loss += loss.item()

    return epoch_loss / len(valid_loader)

for epoch in range(EPOCHS):

    train_loss = train_one_epoch(encoder, decoder, train_loader, optimizer, criterion)
    valid_loss = evaluate_loss(encoder, decoder, valid_loader, criterion)

    print(f"Epoch: {epoch+1:02} | Train Loss: {train_loss:.3f} | Val Loss: {valid_loss:.3f}")

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        print("Saving new best checkpoint!")
        torch.save({
            'encoder_state_dict': encoder.state_dict(),
            'decoder_state_dict': decoder.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, 'best_seq2seq_model.pt')

Epoch: 01 | Train Loss: 4.184 | Val Loss: 3.689
Saving new best checkpoint!
Epoch: 02 | Train Loss: 4.087 | Val Loss: 3.601
Saving new best checkpoint!
Epoch: 03 | Train Loss: 4.013 | Val Loss: 3.568
Saving new best checkpoint!
Epoch: 04 | Train Loss: 3.925 | Val Loss: 3.521
Saving new best checkpoint!
Epoch: 05 | Train Loss: 3.875 | Val Loss: 3.503
Saving new best checkpoint!
Epoch: 06 | Train Loss: 3.816 | Val Loss: 3.484
Saving new best checkpoint!
Epoch: 07 | Train Loss: 3.765 | Val Loss: 3.457
Saving new best checkpoint!
Epoch: 08 | Train Loss: 3.716 | Val Loss: 3.442
Saving new best checkpoint!
Epoch: 09 | Train Loss: 3.654 | Val Loss: 3.451
Epoch: 10 | Train Loss: 3.610 | Val Loss: 3.427
Saving new best checkpoint!
Epoch: 11 | Train Loss: 3.548 | Val Loss: 3.419
Saving new best checkpoint!
Epoch: 12 | Train Loss: 3.507 | Val Loss: 3.417
Saving new best checkpoint!
Epoch: 13 | Train Loss: 3.452 | Val Loss: 3.427
Epoch: 14 | Train Loss: 3.398 | Val Loss: 3.389
Saving new best chec